# SDR cohort overview

Descriptive figure comparing the positive (measured effect) and negative SDR variants: variant counts
and types, distance to the target gene TSS, target genes per variant, CADD, conservation, GTF features,
ENCODE4 cCREs and overlap with cardioid ATAC peaks and ABC candidate enhancers.

**Input:** `VCF_SDR007AB_DLI_SDR010ABC_Comb_Meta.csv` (written by `sdr_variant_dataset.ipynb`) and the
pseudobulk ATAC peaks (`atac_peak_dir`). Saving the figure (to `figures_dir/sdr_cohort`) is commented out.

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from cardiomics_s2f.config import load_paths
from cardiomics_s2f.io.variant_annotation import celltype_from_peak_file, flag_variants_in_peaks, read_narrowpeak
from cardiomics_s2f.vep.cohort_plotting import plot_sdr_cohort_overview

In [ ]:
paths = load_paths()
obs = pd.read_csv(paths.variant_data_dir / "VCF_SDR007AB_DLI_SDR010ABC_Comb_Meta.csv")
FIGURE_DIR = paths.figures_dir / "sdr_cohort"

# Publication style for all figures
sns.set_theme(style="ticks", context="talk", font_scale=0.85)
plt.rcParams['font.family'] = 'sans-serif'

Variants inside and outside the cardioid ATAC peaks of the CM and FB pseudobulks (panels J and K).

In [ ]:
peak_files = glob.glob(os.path.join(paths.atac_peak_dir, "*_peaks.narrowPeak"))
cell_types = {celltype_from_peak_file(p): p for p in peak_files if  celltype_from_peak_file(p) in ['Fibroblasts', 'Cardiomyocytes']}
print("Cell types found:", list(cell_types.keys()))

# flag_variants_in_peaks expects Chromosome, Start and End columns
peak_input = obs.rename(columns={"CHROM": "Chromosome"})

CA_variants = {}
for ct, pf in cell_types.items():
    peaks = read_narrowpeak(pf)
    mask  = flag_variants_in_peaks(peak_input, peaks)
    ct_results = {}
    for cohort, grp in obs[['var_type', "variant_id", "cCRE_class"]].drop_duplicates().groupby("var_type", observed=False):
        grp_mask = mask[grp.index]
        ct_results[cohort] = {
            "in":  grp[grp_mask],
            "out": grp[~grp_mask],
        }
    CA_variants[ct] = ct_results
    print(f"  {ct}: done")

In [ ]:
plot_sdr_cohort_overview(obs[obs.var_type == "positive"],
                         obs[obs.var_type == "negative"],
                         CA_results_dict=CA_variants,
                         celltype_filter=None,
                         # save_fig=True, filepath=FIGURE_DIR / 'VCF_SDR007AB_DLI_SDR010ABC_Comb_Descrip_Ana.png',
                        )